## 4_model_training_pipeline_clarissa_version

My goal with this module is to take the data gathered in gather_historic_data.py script and use it to train a model to predict gas hourly gas burn

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook
from xgboost import XGBRegressor # likely move to modelling module
from sklearn.base import BaseEstimator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


notebook_dir = Path(os.getcwd())
parent_dir = str(notebook_dir.parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
#from scripts.data_pull_functions import login_google_cloud
#from scripts.gather_historic_data import gather_historic_data
from scripts.gather_data_to_forecast import gather_data_to_forecast
from scripts.feature_engineering_functions import add_lag_features, add_rolling_features


In [2]:
model_df = pd.read_csv("../data/processed-data/historic_data_df.csv")
model_df["datetime"] = pd.to_datetime(model_df["datetime"])

forward_df = pd.read_csv("../data/processed-data/data_to_forecast_df.csv")
forward_df["datetime"] = pd.to_datetime(forward_df["datetime"])
forward_df['hourly_gas_burn_MMBtu'] = 0 #placeholder column to assist with concat

In [3]:
print(f'model_df: {model_df.shape}')
print(f'forward_df: {forward_df.shape}')

model_df: (151261, 33)
forward_df: (960, 19)


In [4]:
base_features = ["availability_mw", "load_forecast", "net_load_forecast", "wind_forecast", "temperature_forecast", "wind_speed_forecast", "total_offline_forecast", 
           "offline_ng_forecast", "offline_coal_forecast", "hour", "day_of_week", "month"]

features = base_features + ["gas_lag_1", "gas_lag_24", "gas_lag_168", "gas_roll_24", "gas_roll_168"]


In [5]:
def fit_xgboost_clarissa_version(df: pd.DataFrame, sites = ['CGS', 'DCS', 'GGS', 'LCS', 'PGS'], test_start_date: str='2026-08-27', features: list=[], target: list=[]): 
    #sites could probably reference the constants.py file once it is moved to a .py script
    models = {}
    results = {}
    
    for site in sites:

        site_df = df[df["site"] == site].copy()



        #train on the past and test on the future
        train = site_df[site_df["datetime"] < test_start_date].copy()

        test = site_df[site_df["datetime"] >= test_start_date].copy()



        #Build the model
        model = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=42)


        #Fit the model
        model.fit(train[features], train[target])

        models[site] = model
        
        #generate predictions 
        prediction = model.predict(test[features])


        #evaluate 
        mean_abs_error = mean_absolute_error(test[target], prediction)

        root_mean_squared_error = np.sqrt(mean_squared_error(test[target], prediction))

        r2 = r2_score(test[target], prediction)


        results[site] = {"mae": mean_abs_error, "rmse": root_mean_squared_error, "r2": r2}

    return {'models': models, 'results': results}


In [8]:
fitting_results = fit_xgboost_clarissa_version(df = model_df, sites=['CGS', 'DCS', 'GGS', 'LCS', 'PGS'], test_start_date= '2026-08-20', features = features, target='hourly_gas_burn_MMBtu')
site_models = fitting_results['models']
results = fitting_results['results']

In [133]:
models['CGS']

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


Forecasting Steps:
1. join historic data and future data. Only keep the columns that are in the future df. Keep track of future start date
2. Sort by site, datetime
3. Split by site
    1. run the add_lag and add_rolling features
    2. remove the rows that belong to the historic df
    3. run the predict function
    4. add the prediction back into the DF
    5. recalculate rolling and lag features

In [185]:
# Step 1
full_df = pd.concat([model_df.loc[:, forward_df.columns], forward_df])
full_df.to_csv('../data/processed-data/future_df_check.csv', index=False)
full_df.shape


(152221, 19)

In [186]:
# Step 2
full_df = full_df.sort_values(by = ['site', 'datetime'])
full_df.to_csv('../data/processed-data/future_df_check.csv', index=False)
full_df.shape

(152221, 19)

In [190]:
# Step 3
full_df = add_lag_features(full_df)
full_df = add_rolling_features(full_df)
full_df.to_csv('../data/processed-data/future_df_check.csv', index=False)
full_df.shape


(152221, 24)

In [187]:
future_df = full_df[full_df['datetime']>="2026-08-27"]

In [188]:
forward_dates = future_df['datetime'].unique()
forward_dates


<DatetimeArray>
['2026-08-27 00:00:00', '2026-08-27 01:00:00', '2026-08-27 02:00:00',
 '2026-08-27 03:00:00', '2026-08-27 04:00:00', '2026-08-27 05:00:00',
 '2026-08-27 06:00:00', '2026-08-27 07:00:00', '2026-08-27 08:00:00',
 '2026-08-27 09:00:00',
 ...
 '2026-09-03 14:00:00', '2026-09-03 15:00:00', '2026-09-03 16:00:00',
 '2026-09-03 17:00:00', '2026-09-03 18:00:00', '2026-09-03 19:00:00',
 '2026-09-03 20:00:00', '2026-09-03 21:00:00', '2026-09-03 22:00:00',
 '2026-09-03 23:00:00']
Length: 192, dtype: datetime64[us]

In [ ]:
future_df.to_csv('../data/processed-data/loop_future_df_check.csv', index=False)

In [30]:
def generate_feed_forward_forecast(historic_df: pd.DataFrame=[], forward_df: pd.DataFrame=[], models: list[XGBRegressor]=[], features: list[str]=[], save_output=False):
    """
    Generate sequential forecasts using a feed-forward approach with lag and rolling features.
    
    Combines historic and forecast data, then iteratively generates predictions for future dates.
    For each prediction, recalculates lag and rolling features using the accumulated data (including
    previous predictions) to ensure features reflect the full history up to the prediction point.
    
    Parameters
    ----------
    historic_df : pd.DataFrame, optional
        Dataframe containing historical data with actual observations. Must include columns:
        'site', 'datetime', 'hourly_gas_burn_MMBtu', and all feature columns.
        Default is [].
    forward_df : pd.DataFrame, optional
        Dataframe with future dates and feature values to generate forecasts for.
        Must include 'site', 'datetime', and all feature columns. Default is [].
    models : list[XGBRegressor], optional
        Dictionary mapping site identifiers to fitted XGBRegressor model objects.
        Default is [].
    features : list[str], optional
        List of feature column names used as input to the models. Default is [].
    
    Returns
    -------
    pd.DataFrame
        Dataframe with predicted gas burn values containing columns:
        - 'site': Site identifier
        - 'datetime': Prediction datetime
        - 'gasday': Gas day (calculated as datetime minus 10 hours)
        - 'hourly_gas_burn_MMBtu': Predicted hourly gas burn value
    
    Notes
    -----
    Predictions are set to 0 if availability is <= 1 MW.
    Lag and rolling features are recalculated for each prediction to incorporate previous predictions
    in the feature values, enabling truly forward-looking forecasts.
    Uses a windowed approach (up to 336 hours/14 days of recent history) for efficiency.
    """

    predictions_by_site = []
    sites = historic_df['site'].unique()
    forward_dates = forward_df['datetime'].unique()
    full_df = pd.concat([historic_df, forward_df])

    for site in sites:
        site_model = models[site]

        # filter to site and set datetime as the index for quicker look up
        site_full_df = full_df.loc[full_df['site']==site, ['site','datetime','hourly_gas_burn_MMBtu'] + features]
        site_full_df = site_full_df.set_index('datetime')
        site_full_df = site_full_df.sort_index()


        # only looping through future dates
        for date in forward_dates:

            # need to calculate the lag and rolling features each time a new datapoint (i.e. prediction) is added the site_full_df
            # Want to avoid updating the entire dataframe, so going to only update a 'window' of it using only the most recent records that are needed to get rolling/lag values from up to 2 weeks ago
            current_index = site_full_df.index.get_loc(date)
            start_index = max(0, current_index-336)
            window_df = site_full_df.iloc[start_index: current_index + 1].copy()

            window_df = add_lag_features(window_df)
            window_df = add_rolling_features(window_df)

            # Need to replace the current values with the updated window versions
            for col in features:
                if col in window_df.columns:
                    site_full_df.loc[date, col] = window_df.loc[date, col]


            # Limiting to only the single datetime that is to be predicted. If the availability for that time <1 set forecast to 0
            record_to_predict = site_full_df.loc[[date], features]
            prediction =  0 if record_to_predict['availability_mw'].values <=1 else site_model.predict(record_to_predict)[0]


            prediction_row = {
                'site': site,
                'datetime': date,
                'gasday': (pd.to_datetime(date)- pd.Timedelta(hours=10)).date(),
                'hourly_gas_burn_MMBtu': prediction 
            }

            # feeding the predicted gas burn back into the site df
            # The predicted value will then be fed forward to calculate the lag and rolling values for the next future time period
            site_full_df.loc[date, 'hourly_gas_burn_MMBtu'] = prediction_row['hourly_gas_burn_MMBtu']
            

            predictions_by_site.append(prediction_row)

    predictions_by_site_df = pd.DataFrame(predictions_by_site)

    ##################
    ## Confirmation ##
    ##################
    if save_output: 
        folder = Path("../data/output-data/forecasts/")
        if not folder.exists():
            folder.mkdir(parents=True, exist_ok=True)
        file_date = datetime.now().date()
        file_name = folder / f'forecasts_{file_date}.csv'
        predictions_by_site_df.to_csv(file_name, index=False)
        print(f'A file containing the future data to has been saved to {file_name}')

    return predictions_by_site_df

In [32]:
predictions = generate_feed_forward_forecast(historic_df=model_df, forward_df=forward_df, models=site_models, features=features, save_output=True)

A file containing the future data to has been saved to ..\data\output-data\forecasts\forecasts_2026-08-28.csv


In [33]:
predictions[predictions['site']=='PGS'].head(24)['hourly_gas_burn_MMBtu'].sum()

np.float64(159561.3740234375)

In [ ]:
all_forecasts = []

def forecast_future_clarissa(df: pd.DataFrame=[], models=[BaseEstimator]):

    for site in models:

        print(f"Forecasting for {site}")

        model = models[site]


        # Future hours for this site
        site_future = (df[forward_df["site"] == site].sort_values("datetime").copy())



        # Last 168 actual hours used to seed lags
        history = (model_df[model_df["site"] == site].sort_values("datetime").tail(168).copy())

        for _, row in site_future.iterrows():

            row = row.copy()


        # Create lag features


            row["gas_lag_1"] = (history["hourly_gas_burn_MMBtu"].iloc[-1])

            row["gas_lag_24"] = (history["hourly_gas_burn_MMBtu"].iloc[-24])

            row["gas_lag_168"] = (history["hourly_gas_burn_MMBtu"].iloc[-168])

            row["gas_roll_24"] = (history["hourly_gas_burn_MMBtu"].tail(24).mean())

            row["gas_roll_168"] = (history["hourly_gas_burn_MMBtu"].tail(168).mean())





        # Make prediction
            X_pred = pd.DataFrame([row])[features]

            if row["availability_mw"] <= 1:

                prediction = 0
                
            else:

                prediction = model.predict(X_pred)[0]
                
            prediction = max(0, prediction)
            



        # Save forecast

            all_forecasts.append({"datetime": row["datetime"], "site": site, "predicted_gas_burn_MMBtu": prediction})

            forecast_df = pd.DataFrame(all_forecasts)




        # Add prediction back into history for future lag values

        history = pd.concat([history, pd.DataFrame([{"datetime": row["datetime"], "hourly_gas_burn_MMBtu": prediction}])], ignore_index=True)

        # Keep most recent 168 hours (7 days)
        history = history.tail(168)

### Formatting for NBPL Excel file


In [54]:
daily_record_counts = predictions.groupby('gasday').count()['site'].reset_index()

full_gas_days = daily_record_counts[daily_record_counts['site']==max(daily_record_counts['site'])]['gasday']
full_gas_days = pd.to_datetime(full_gas_days)
#(predictions.groupby('gasday').count()['site'] == full_day_count).index

In [55]:
full_gas_days

1   2026-08-27
2   2026-08-28
3   2026-08-29
4   2026-08-30
5   2026-08-31
6   2026-09-01
7   2026-09-02
Name: gasday, dtype: datetime64[s]

In [60]:
file_date = min(full_gas_days).date()
print(file_date)

2026-08-27


In [100]:
def create_nbpl_file(predictions: pd.DataFrame, file_date: str=None):
    """
    
    """

    # dynamically getting date based on first day with full gas day predictions
    if not file_date:
        daily_record_counts = predictions.groupby('gasday').count()['site'].reset_index()

        full_gas_days = daily_record_counts[daily_record_counts['site']==max(daily_record_counts['site'])]['gasday']
        full_gas_days = pd.to_datetime(full_gas_days)
        print(full_gas_days)
        file_date = min(full_gas_days).date()

    
    template_file = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Forecast template no links.xlsx"

    workbook = load_workbook(template_file, keep_vba=True, data_only=True)

    worksheet = workbook["Daily Burn Sheet"]

            
    # updating start and end date in the file
    worksheet["D11"] = file_date
    worksheet["F11"] = file_date
    
          
    site_to_columns = {'CGS': 6, 'DCS': 8, 'GGS': 10, 'LCS': 12, 'PGS': 14}
        
    for site, col in site_to_columns.items():

        site_predictions = predictions.loc[(predictions['site']==site) & (predictions['gasday']==file_date), 'hourly_gas_burn_MMBtu']
        for row in range(0, 24):
            
            excel_row = 30 + row
            #print(f'col {col}   row {row}   excel row {excel_row}')
            worksheet.cell(row=excel_row, column=col, value=site_predictions.values[row])   # F
            #worksheet.cell(row=excel_row, column=8, value=row["DCS"])   # H
            #worksheet.cell(row=excel_row, column=10, value=row["GGS"])  # J
            #worksheet.cell(row=excel_row, column=12, value=row["LCS"])  # L
            #worksheet.cell(row=excel_row, column=14, value=row["PGS"])  # N


    today_str = datetime.today().strftime("%m.%d.%Y")

    output_file = (rf"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Hourly Gas Burn Forecast for {today_str}.xlsm")

    workbook.save(output_file)

    print(output_file)

    return file_date

In [101]:
create_nbpl_file(predictions=predictions)

1   2026-08-27
2   2026-08-28
3   2026-08-29
4   2026-08-30
5   2026-08-31
6   2026-09-01
7   2026-09-02
Name: gasday, dtype: datetime64[s]
G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Hourly Gas Burn Forecast for 08.28.2026.xlsm


datetime.date(2026, 8, 27)

In [ ]:
# Create NBPL layout
format = (forecast_df.pivot(index="datetime", columns="site", values="predicted_gas_burn_MMBtu").reset_index())

# Only first 24 hours belong on the burn sheet
format = format.head(24)

print(format.head())
#change this to a download the user has

template_file = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Forecast template.xlsx"

workbook = load_workbook(template_file, keep_vba=True)

worksheet = workbook["Daily Burn Sheet"]

        
gas_day = format["datetime"].min().date()

worksheet["D11"] = gas_day
worksheet["F11"] = gas_day
 
        
for i, (_, row) in enumerate(format.iterrows()):

    excel_row = 30 + i

    worksheet.cell(row=excel_row, column=6, value=row["CGS"])   # F
    worksheet.cell(row=excel_row, column=8, value=row["DCS"])   # H
    worksheet.cell(row=excel_row, column=10, value=row["GGS"])  # J
    worksheet.cell(row=excel_row, column=12, value=row["LCS"])  # L
    worksheet.cell(row=excel_row, column=14, value=row["PGS"])  # N



today_str = datetime.today().strftime("%m.%d.%Y")

output_file = (rf"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Forecasts\Hourly Gas Burn Forecast for {today_str}.xlsm")

workbook.save(output_file)

print(output_file)